In [98]:
import sys
sys.path.append('../')
import numpy as np
import torch
import torch.nn.functional as F
from mingpt.model import GPT
import pickle


In [99]:
experiment_save_folder = '../experiments/'
N_values = [10, 25, 50, 100, 200, 300, 499]

In [100]:

def iid_generate(p_dict, N):
    p0 = p_dict[0]
    p1 = p_dict[1]
    return np.random.choice([0,1], size=N, p=[p0, p1])

def generate_bin_pmf():
    p = np.random.rand()
    dict_out = {
        0: p,
        1: 1-p
    }
    return dict_out

def convert_sample_to_pmf(sample):
    values, counts = np.unique(sample, return_counts=True)
    probs = counts/counts.sum()
    return dict(zip(values, probs))

def entropy_calc(pmf):
    pmf = clean_pmf(pmf)
    pmf_array = np.asarray(list(pmf.values()))
    return -np.sum(pmf_array * np.log2(pmf_array))

def clean_pmf(pmf):
    return {k: v for k, v in pmf.items() if v > 0.0}

In [101]:
def sequential_universal_source_coding(seq, p_array = None):
    total_bits = 0
    k = 0
    i = 0

    for bit in seq:
        if p_array is not None:
            p1 = p_array[i]
        else:
            p1 = (k+1) / (i+2)
        p0 = 1-p1

        if bit == 1:
            total_bits += -np.log2(p1)
        else:
            total_bits += -np.log2(p0)
        k += bit
        i += 1

    bits_per_symbol = total_bits / len(seq)
    return bits_per_symbol

In [102]:

# generate an array of probabilities given a bit sequence

def transformer_p_array(sequence, model, device='mps'):
    model.eval()
    sequence=torch.tensor(sequence, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        logits, _ = model(sequence)  # [1, T, vocab_size]
        # convert to probabilities
        probs = F.softmax(logits, dim=-1)  # [1, T, vocab_size]
        # extract probability of token=1 at each timestep
        p_array = [0.5] + probs[0, :-1, 1].tolist()
    
    return p_array

In [103]:
def load_model(model_dir):
    checkpoint = torch.load(model_dir, map_location='cpu')
    block_size = checkpoint['block_size']
    
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = block_size
    
    model = GPT(model_config)
    model.load_state_dict(checkpoint['state_dict'])
    model = model.to('mps')
    model.eval()
    return model

In [ ]:
def markov_generate(p_stay, N):
    state = np.random.choice([0, 1])  # stationary is uniform for symmetric chain
    seq = [state]
    for _ in range(N - 1):
        if np.random.rand() < p_stay:
            seq.append(state)  # stay
        else:
            state = 1 - state  # switch
            seq.append(state)
    return np.array(seq)


In [105]:
# Transformer vs Laplace prior quality at small N

def run_experiment(model, p_values, N_values, output_file, n_trials=100, device='mps'):
    """
    For each (p, N) pair, run n_trials with identical sequences for both methods.
    Returns nested dict: results[p][N] = {'laplace': (mean, std), 'transformer': (mean, std)}
    """
    all_results = {}
    save_dir = experiment_save_folder + output_file
    print(f'Saving to {save_dir}')

    for p_true in p_values:
        print(f"\nRunning p={p_true}")
        p_dict = {0: 1 - p_true, 1: p_true}
        all_results[p_true] = {}

        for N in N_values:
            laplace_bps_trials = []
            transformer_bps_trials = []

            for trial in range(n_trials):
                # Same sequence for both methods
                seq = iid_generate(p_dict, N)

                # Laplace
                bps_lap = sequential_universal_source_coding(seq)
                laplace_bps_trials.append(bps_lap)

                # Transformer
                p_array = transformer_p_array(seq, model, device=device)
                bps_trans = sequential_universal_source_coding(seq, p_array=p_array)
                transformer_bps_trials.append(bps_trans)

            all_results[p_true][N] = {
                'laplace': (np.mean(laplace_bps_trials), np.std(laplace_bps_trials)),
                'transformer': (np.mean(transformer_bps_trials), np.std(transformer_bps_trials)),
                'entropy': entropy_calc({p_true: p_true, 1-p_true: 1-p_true})
            }
            print(f"  N={N}: laplace={np.mean(laplace_bps_trials):.4f}, transformer={np.mean(transformer_bps_trials):.4f}")

    with open(save_dir, 'wb') as f:
        pickle.dump(all_results, f)

    print(f'Saved experiment to {save_dir}')




In [106]:
def run_markov_experiment(model, p_stay_values, N_values, output_file, n_trials=100, device='mps'):
    all_results = {}
    save_dir = experiment_save_folder + output_file
    print(f'Saving to {save_dir}')
    
    for p_stay in p_stay_values:
        print(f"\nRunning p_stay={p_stay}")
        all_results[p_stay] = {}
        
        for N in N_values:
            laplace_bps_trials = []
            transformer_bps_trials = []
            
            for trial in range(n_trials):
                seq = markov_generate(p_stay, N)
                laplace_bps_trials.append(sequential_universal_source_coding(seq))
                p_array = transformer_p_array(seq, model, device=device)
                transformer_bps_trials.append(sequential_universal_source_coding(seq, p_array=p_array))
            
            all_results[p_stay][N] = {
                'laplace': (np.mean(laplace_bps_trials), np.std(laplace_bps_trials)),
                'transformer': (np.mean(transformer_bps_trials), np.std(transformer_bps_trials)),
            }
            print(f"  N={N}: laplace={np.mean(laplace_bps_trials):.4f}, transformer={np.mean(transformer_bps_trials):.4f}")
    
    with open(save_dir, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'Saved experiment to {save_dir}')

In [107]:

model = load_model('../models/iid_model.pt')

compress_seq = iid_generate(generate_bin_pmf(), 99)  # 99 because model block_size = n-1 = 99
generated_p = convert_sample_to_pmf(compress_seq)

p_array = transformer_p_array(compress_seq.tolist(), model)
p_array = [0.5] + p_array[:-1]  # assume first bit is uniform (no prior)
bps_transformer = sequential_universal_source_coding(compress_seq, p_array=p_array)

bps_laplace = sequential_universal_source_coding(compress_seq)
bps_entropy = entropy_calc(generated_p)

print(f'Entropy (lower bound): {bps_entropy:.4f}')
print(f'Laplace BPS:           {bps_laplace:.4f}')
print(f'Transformer BPS:       {bps_transformer:.4f}')

number of parameters: 0.11M
Entropy (lower bound): 0.4050
Laplace BPS:           0.4440
Transformer BPS:       0.4486


In [108]:
model_iid = load_model('../models/iid_model.pt')

p_values_iid = [0.1, 0.3, 0.5, 0.7, 0.9]

run_experiment(model_iid, p_values_iid, N_values, output_file='iid_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ../experiments/iid_experiment.pkl

Running p=0.1
  N=10: laplace=0.6085, transformer=0.6116
  N=25: laplace=0.5436, transformer=0.5460
  N=50: laplace=0.5116, transformer=0.5135
  N=100: laplace=0.5092, transformer=0.5111
  N=200: laplace=0.4794, transformer=0.4803
  N=300: laplace=0.4782, transformer=0.4790
  N=499: laplace=0.4849, transformer=0.4854

Running p=0.3
  N=10: laplace=0.9729, transformer=0.9810
  N=25: laplace=0.9409, transformer=0.9465
  N=50: laplace=0.9077, transformer=0.9116
  N=100: laplace=0.9144, transformer=0.9165
  N=200: laplace=0.8888, transformer=0.8898
  N=300: laplace=0.8958, transformer=0.8965
  N=499: laplace=0.8894, transformer=0.8900

Running p=0.5
  N=10: laplace=1.0816, transformer=1.0910
  N=25: laplace=1.0528, transformer=1.0578
  N=50: laplace=1.0399, transformer=1.0417
  N=100: laplace=1.0221, transformer=1.0234
  N=200: laplace=1.0126, transformer=1.0132
  N=300: laplace=1.0099, transformer=1.0105
  N=499: lap

In [109]:
model_p03 = load_model('../models/p03_model.pt')

p_values_03 = [0.3]

run_experiment(model_p03, p_values_03, N_values, output_file='p03_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ../experiments/p03_experiment.pkl

Running p=0.3
  N=10: laplace=0.9744, transformer=0.9140
  N=25: laplace=0.9282, transformer=0.8748
  N=50: laplace=0.9319, transformer=0.8978
  N=100: laplace=0.9067, transformer=0.8837
  N=200: laplace=0.8994, transformer=0.8866
  N=300: laplace=0.8923, transformer=0.8820
  N=499: laplace=0.8897, transformer=0.8826
Saved experiment to ../experiments/p03_experiment.pkl


In [110]:

model_p07 = load_model('../models/p07_model.pt')

p_values_07 = [0.7]

run_experiment(model_p07, p_values_07, N_values, output_file='p07_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ../experiments/p07_experiment.pkl

Running p=0.7
  N=10: laplace=0.9588, transformer=0.8895
  N=25: laplace=0.9326, transformer=0.8807
  N=50: laplace=0.9335, transformer=0.8962
  N=100: laplace=0.8974, transformer=0.8751
  N=200: laplace=0.8966, transformer=0.8835
  N=300: laplace=0.8994, transformer=0.8885
  N=499: laplace=0.8883, transformer=0.8812
Saved experiment to ../experiments/p07_experiment.pkl


In [111]:
model_points = load_model('../models/points_model.pt')

p_values_points = [0.1, 0.3, 0.5, 0.7, 0.9]

run_experiment(model_points, p_values_points, N_values, output_file='points_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ../experiments/points_experiment.pkl

Running p=0.1
  N=10: laplace=0.6346, transformer=0.6198
  N=25: laplace=0.5652, transformer=0.5492
  N=50: laplace=0.5089, transformer=0.4990
  N=100: laplace=0.5025, transformer=0.4939
  N=200: laplace=0.4872, transformer=0.4797
  N=300: laplace=0.4814, transformer=0.4768
  N=499: laplace=0.4781, transformer=0.4747

Running p=0.3
  N=10: laplace=0.9859, transformer=0.9878
  N=25: laplace=0.9311, transformer=0.9347
  N=50: laplace=0.9229, transformer=0.9244
  N=100: laplace=0.9013, transformer=0.9012
  N=200: laplace=0.8964, transformer=0.8954
  N=300: laplace=0.8932, transformer=0.8917
  N=499: laplace=0.8907, transformer=0.8899

Running p=0.5
  N=10: laplace=1.0570, transformer=1.0667
  N=25: laplace=1.0539, transformer=1.0629
  N=50: laplace=1.0350, transformer=1.0415
  N=100: laplace=1.0213, transformer=1.0252
  N=200: laplace=1.0142, transformer=1.0151
  N=300: laplace=1.0099, transformer=1.0106
  N=499: 

In [113]:
p_stay_sweep = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99]
N_values = [10, 25, 50, 100, 200, 300, 499]

model_iid = load_model('../models/iid_model.pt')
run_markov_experiment(model_iid, p_stay_sweep, N_values, output_file='iid_experiment_markov.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ../experiments/iid_experiment_markov.pkl

Running p_stay=0.0
  N=10: laplace=1.1437, transformer=1.1566
  N=25: laplace=1.0804, transformer=1.0763
  N=50: laplace=1.0504, transformer=1.0472
  N=100: laplace=1.0301, transformer=1.0326
  N=200: laplace=1.0175, transformer=1.0256
  N=300: laplace=1.0126, transformer=1.0237
  N=499: laplace=1.0083, transformer=1.0227

Running p_stay=0.3
  N=10: laplace=1.1187, transformer=1.1291
  N=25: laplace=1.0679, transformer=1.0697
  N=50: laplace=1.0448, transformer=1.0451
  N=100: laplace=1.0277, transformer=1.0298
  N=200: laplace=1.0158, transformer=1.0192
  N=300: laplace=1.0117, transformer=1.0164
  N=499: laplace=1.0077, transformer=1.0135

Running p_stay=0.5
  N=10: laplace=1.0838, transformer=1.0915
  N=25: laplace=1.0480, transformer=1.0513
  N=50: laplace=1.0343, transformer=1.0374
  N=100: laplace=1.0227, transformer=1.0244
  N=200: laplace=1.0142, transformer=1.0150
  N=300: laplace=1.0102, transform